In [1]:
import pandas as pd
import re

print("Libraries Loaded")

Libraries Loaded


In [2]:
df = pd.read_parquet(
    "../../data/processed/classification_dataset_final.parquet"
)

print(df.shape)

df.head()

(26688, 4)


,case_name,year,legal_summary,category
0,A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_In...,1950,The petitioner who was detained under the Pre...,Constitutional
1,A_M_Mair_Co_vs_Gordhandass_Sagarmull_on_30_Nov...,1950,A.M. Mair & Co vs Gordhandass Sagarmull on 30...,Contract
2,Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14...,1950,"The appellant, an estate broker, was employed...",Property
3,Arjuna_Lal_Misra_vs_The_State_on_30_November_1...,1950,"Fazl Ali, J. 1. I do not wish to express diss...",Criminal
4,Ashutosh_Lahiry_vs_The_State_Of_Delhi_And_Anr_...,1950,"""You came to Delhi on March 27, 1950 and held...",Civil


In [3]:
print(df.columns)

Index(['case_name', 'year', 'legal_summary', 'category'], dtype='object')


In [4]:
def extract_articles(text):

    if pd.isna(text):
        return []

    pattern = r'Article\s+\d+[A-Za-z()0-9]*'

    matches = re.findall(
        pattern,
        str(text),
        flags=re.IGNORECASE
    )

    cleaned = []

    for m in matches:

        article = (
            m.replace("article", "Article")
             .strip()
        )

        cleaned.append(article)

    return list(set(cleaned))

In [5]:
df["articles"] = df["legal_summary"].apply(
    extract_articles
)

print(
    df["articles"].head()
)

0    [Article 19]
1              []
2              []
3              []
4              []
Name: articles, dtype: object


In [6]:
article_df = df[
    df["articles"].apply(
        lambda x: len(x) > 0
    )
].copy()

print(article_df.shape)

(2334, 5)


In [7]:
article_df[
    ["case_name","articles"]
].head(20)

,case_name,articles
0,A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_In...,[Article 19]
12,Co_Operative_Society_Of_Debts_vs_Nandlal_on_12...,[article 374(4)]
29,Jeevantha_And_Ors_vs_Hanumantha_And_Ors_on_20_...,[Article 374(4)]
36,Naguba_Appa_vs_Namdev_on_20_November_1950_1,[Article 374(4)]
89,In_Re__The_Delhi_Laws_Act_1912_The_vs_Unknown_...,[article 143]
93,Joylal_Agarwala_vs_The_State_on_4_October_1951_1,"[article 136(1), article 134(1)(c)]"
97,Keshavan_Madhava_Menon_vs_The_State_Of_Bombay_...,[Article 13(1)]
109,P_D_Shamdasani_vs_Central_Bank_Of_India_Ltd_on...,"[Article 19(1)(f), article 32]"
113,Raja_Braja_Sundar_Deb_vs_Moni_Behara_And_Other...,[article 47]
116,Ram_Singh_And_Ors_vs_The_State_Of_Delhi_And_An...,"[article 32, article 226]"


In [7]:
article_df = article_df.explode(
    "articles"
)

article_df = article_df.rename(
    columns={
        "articles": "article"
    }
)

print(article_df.shape)

article_df.head()

(2983, 5)


,case_name,year,legal_summary,category,article
0,A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_In...,1950,The petitioner who was detained under the Pre...,Constitutional,Article 19
12,Co_Operative_Society_Of_Debts_vs_Nandlal_on_12...,1950,"Mahajan, J. 1. This appeal arises out of exec...",Constitutional,Article 374(4)
29,Jeevantha_And_Ors_vs_Hanumantha_And_Ors_on_20_...,1950,"Mahajan, J. 1. These two appeals were present...",Constitutional,Article 374(4)
36,Naguba_Appa_vs_Namdev_on_20_November_1950_1,1950,"Mahajan, J. 1. A decree for pre-emption of th...",Constitutional,Article 374(4)
89,In_Re__The_Delhi_Laws_Act_1912_The_vs_Unknown_...,1951,"Kania, C.J. 1. This is a reference made by th...",Constitutional,Article 143


In [9]:
article_df.to_parquet(
    "../../data/processed/article_prediction_dataset.parquet",
    index=False
)

print(
    article_df.shape
)

(2334, 5)


In [10]:
top_articles.head(20)

,article,count
36,Article 226,477
15,Article 32,456
32,Article 136,220
40,Article 14,173
100,Article 227,88
77,Article 309,71
149,Article 21,70
111,Article 133(1)(a),36
10,article 226,36
89,Article 22(5),35


In [8]:
print(
    article_df["article"]
    .value_counts()
    .head(20)
)

article
Article 226          513
Article 32           477
Article 136          228
Article 14           186
Article 227           88
Article 21            75
Article 309           72
Article 133(1)(a)     37
Article 19(1)(g)      36
Article 22(5)         35
Article 12            32
Article 133           32
Article 311(2)        31
Article 311           30
Article 19            26
Article 16            23
Article 22            22
Article 134(1)(c)     20
Article 137           18
Article 142           17
Name: count, dtype: int64


In [9]:
article_df[
    [
        "legal_summary",
        "article"
    ]
].to_parquet(
    "../../data/processed/article_prediction_dataset.parquet",
    index=False
)

print("Dataset Saved")

Dataset Saved


In [10]:
final_df = pd.read_parquet(
    "../../data/processed/article_prediction_dataset.parquet"
)

print(final_df.shape)

final_df.head()

(2983, 2)


,legal_summary,article
0,The petitioner who was detained under the Pre...,Article 19
1,"Mahajan, J. 1. This appeal arises out of exec...",Article 374(4)
2,"Mahajan, J. 1. These two appeals were present...",Article 374(4)
3,"Mahajan, J. 1. A decree for pre-emption of th...",Article 374(4)
4,"Kania, C.J. 1. This is a reference made by th...",Article 143


In [11]:
print(article_df.shape)

print(
    article_df["article"]
    .value_counts()
    .head(20)
)

(2983, 5)
article
Article 226          513
Article 32           477
Article 136          228
Article 14           186
Article 227           88
Article 21            75
Article 309           72
Article 133(1)(a)     37
Article 19(1)(g)      36
Article 22(5)         35
Article 12            32
Article 133           32
Article 311(2)        31
Article 311           30
Article 19            26
Article 16            23
Article 22            22
Article 134(1)(c)     20
Article 137           18
Article 142           17
Name: count, dtype: int64
